In [11]:
import os
from google.colab import drive
drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/HPC')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
!uname -a && cat /etc/*release

Linux 865e72b265d2 6.6.122+ #1 SMP Thu Apr 30 18:17:14 UTC 2026 x86_64 x86_64 x86_64 GNU/Linux
DISTRIB_ID=Ubuntu
DISTRIB_RELEASE=22.04
DISTRIB_CODENAME=jammy
DISTRIB_DESCRIPTION="Ubuntu 22.04.5 LTS"
PRETTY_NAME="Ubuntu 22.04.5 LTS"
NAME="Ubuntu"
VERSION_ID="22.04"
VERSION="22.04.5 LTS (Jammy Jellyfish)"
VERSION_CODENAME=jammy
ID=ubuntu
ID_LIKE=debian
HOME_URL="https://www.ubuntu.com/"
SUPPORT_URL="https://help.ubuntu.com/"
BUG_REPORT_URL="https://bugs.launchpad.net/ubuntu/"
PRIVACY_POLICY_URL="https://www.ubuntu.com/legal/terms-and-policies/privacy-policy"
UBUNTU_CODENAME=jammy


In [5]:
!pwd
!ls -la

/content
total 16
drwxr-xr-x 1 root root 4096 Jun  4 13:32 .
drwxr-xr-x 1 root root 4096 Jun 27 12:58 ..
drwxr-xr-x 4 root root 4096 Jun  4 13:32 .config
drwxr-xr-x 1 root root 4096 Jun  4 13:32 sample_data


In [6]:
!lscpu

Architecture:                x86_64
  CPU op-mode(s):            32-bit, 64-bit
  Address sizes:             46 bits physical, 48 bits virtual
  Byte Order:                Little Endian
CPU(s):                      2
  On-line CPU(s) list:       0,1
Vendor ID:                   GenuineIntel
  Model name:                Intel(R) Xeon(R) CPU @ 2.20GHz
    CPU family:              6
    Model:                   79
    Thread(s) per core:      2
    Core(s) per socket:      1
    Socket(s):               1
    Stepping:                0
    BogoMIPS:                4399.99
    Flags:                   fpu vme de pse tsc msr pae mce cx8 apic sep mtrr pg
                             e mca cmov pat pse36 clflush mmx fxsr sse sse2 ss h
                             t syscall nx pdpe1gb rdtscp lm constant_tsc rep_goo
                             d nopl xtopology nonstop_tsc cpuid tsc_known_freq p
                             ni pclmulqdq ssse3 fma cx16 pcid sse4_1 sse4_2 x2ap
                   

In [1]:
!nvidia-smi

Sat Jun 27 13:50:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
%%writefile matmul.c
#include <stdio.h>
#include <stdlib.h>
#include <omp.h>

int main(int argc, char **argv) {
    if (argc < 2) {
        printf("Error: specify the dimension N.\n");
        printf("Usage: %s <N>\n", argv[0]);
        return 1;
    }

    int n = atoi(argv[1]);
    int i, j, k;

    double (*a)[n] = malloc(sizeof(double[n][n]));
    double (*b)[n] = malloc(sizeof(double[n][n]));
    double (*c)[n] = malloc(sizeof(double[n][n]));

    if (!a || !b || !c) {
        printf("Memory allocation error for N=%d\n", n);
        return 1;
    }
    for (i = 0; i < n; i++) {
        for (j = 0; j < n; j++) {
            a[i][j] = 2.0;
            b[i][j] = 3.0;
            c[i][j] = 0.0;
        }
    }

    double start_time = omp_get_wtime();

    for (i = 0; i < n; ++i) {
        for (k = 0; k < n; k++) {
            for (j = 0; j < n; ++j) {
                c[i][j] += a[i][k] * b[k][j];
            }
        }
    }

    double run_time = omp_get_wtime() - start_time;
    printf("Computation time (N=%d): %f seconds\n", n, run_time);

    #ifdef DEBUG
    FILE *f = fopen("mat-res.txt", "w");
    if (!f) {
        perror("fopen error");
        return 1;
    }

    fprintf(f, "%d\n\n", n);

    int print_limit = (n < 1000) ? n : 1000;

    for (int i = 0; i < print_limit; i++) {
        for (int j = 0; j < print_limit; j++) {
            fprintf(f, "%.0f ", c[i][j]);
        }
        fprintf(f, "\n");
    }
    fclose(f);
    printf("Debug file 'mat-res.txt' successfully saved.\n");
    #endif
    free(a);
    free(b);
    free(c);

    return 0;
}

Writing matmul.c


In [6]:
!gcc -O3 -fopenmp matmul.c -o matmul_seq



In [7]:
!time ./matmul_seq 5000

Computation time (N=5000): 107.806994 seconds

real	1m48.142s
user	1m46.953s
sys	0m0.376s


In [ ]:
%%writefile matmul_cuda.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

// ---------------------------------------------------------
// GPU KERNEL: Executed on the graphics card
// ---------------------------------------------------------
__global__ void matMulKernel(double *a, double *b, double *c, int n) {
    // 1. Calculate the global 2D coordinates of the thread
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    int row = blockIdx.y * blockDim.y + threadIdx.y;

    // 2. Check that the thread is within the matrix boundaries
    if (row < n && col < n) {
        double sum = 0.0;

        // 3. The actual computation (The only remaining loop!)
        // Notice how we access the 1D array simulating a 2D matrix (row * n + col)
        for (int k = 0; k < n; k++) {
            sum += a[row * n + k] * b[k * n + col];
        }

        // 4. Save the result
        c[row * n + col] = sum;
    }
}

// ---------------------------------------------------------
// MAIN HOST: Executed on the CPU
// ---------------------------------------------------------
int main(int argc, char **argv) {
    if (argc < 2) {
        printf("Error: specify the dimension N.\n");
        return 1;
    }
    int n = atoi(argv[1]);
    size_t size = (size_t)n * n * sizeof(double);

    // --- HOST MEMORY ALLOCATION (CPU) ---
    // We use "flattened" 1D arrays because they are the safest for GPU transfers
    double *h_a = (double *)malloc(size);
    double *h_b = (double *)malloc(size);
    double *h_c = (double *)malloc(size);

    if (!h_a || !h_b || !h_c) {
        printf("Host RAM allocation error for N=%d\n", n);
        return 1;
    }

    // Data initialization on the CPU
    for (int i = 0; i < n * n; i++) {
        h_a[i] = 2.0;
        h_b[i] = 3.0;
        h_c[i] = 0.0;
    }

    // --- PREPARATION OF CUDA TIMERS ---
    cudaEvent_t start, stop_h2d, stop_kernel, stop_d2h;
    cudaEventCreate(&start);
    cudaEventCreate(&stop_h2d);
    cudaEventCreate(&stop_kernel);
    cudaEventCreate(&stop_d2h);

    // --- DEVICE MEMORY ALLOCATION (GPU) ---
    double *d_a, *d_b, *d_c;
    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    // =======================================================
    // 1. DATA TRANSFER: CPU -> GPU (HostToDevice)
    // =======================================================
    cudaEventRecord(start); // Start the timer

    cudaMemcpy(d_a, h_a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, h_b, size, cudaMemcpyHostToDevice);

    cudaEventRecord(stop_h2d); // Record end of H2D

    // =======================================================
    // 2. KERNEL EXECUTION
    // =======================================================
    // Define the magic block in multiples of 32 (256 total threads)
    dim3 threadsPerBlock(16, 16);
    // Calculate the grid by rounding up to cover the entire matrix
    dim3 numBlocks((n + 15) / 16, (n + 15) / 16);

    matMulKernel<<<numBlocks, threadsPerBlock>>>(d_a, d_b, d_c, n);

    cudaEventRecord(stop_kernel); // Record end of GPU computation

    // =======================================================
    // 3. RESULT TRANSFER: GPU -> CPU (DeviceToHost)
    // =======================================================
    cudaMemcpy(h_c, d_c, size, cudaMemcpyDeviceToHost);

    cudaEventRecord(stop_d2h); // Record end of D2H
    cudaEventSynchronize(stop_d2h); // Wait for everything to completely finish

    // --- CALCULATION AND PRINTING OF TIMES ---
    float ms_h2d, ms_kernel, ms_d2h;
    cudaEventElapsedTime(&ms_h2d, start, stop_h2d);
    cudaEventElapsedTime(&ms_kernel, stop_h2d, stop_kernel);
    cudaEventElapsedTime(&ms_d2h, stop_kernel, stop_d2h);

    printf("Matrix Dimension N           : %d\n", n);
    printf("Transfer Time CPU -> GPU     : %f ms\n", ms_h2d);
    printf("GPU Kernel Computation Time  : %f ms\n", ms_kernel);
    printf("Transfer Time GPU -> CPU     : %f ms\n", ms_d2h);
    printf("Total Time (Data + Compute)  : %f ms\n", ms_h2d + ms_kernel + ms_d2h);

    // --- MEMORY CLEANUP ---
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    free(h_a); free(h_b); free(h_c);

    // Destroy the timers
    cudaEventDestroy(start); cudaEventDestroy(stop_h2d);
    cudaEventDestroy(stop_kernel); cudaEventDestroy(stop_d2h);

    return 0;
}

In [ ]:
!nvcc -arch=sm_75 -O3 matmul.cu -o matmul_gpu

In [ ]:
!nsys profile -t cuda,osrt -o /content/drive/MyDrive/report_matmul --stats=true --force-overwrite=true ./matmul_gpu 5000

In [2]:
!/content/matmul_gpu 5000

/bin/bash: line 1: /content/test: No such file or directory


In [ ]:
!nvprof /content/matmul_gpu 5000

==5746== NVPROF is profiling process 5746, command: /content/test

--- DEVICE 0: "Tesla T4" ---
  CUDA Capability Major/Minor version:          7.5
  Total amount of global memory:                14.56 GBytes
  Number of multiprocessors (SMs):              40
  Total amount of constant memory:              65536 bytes
  Total amount of shared memory per block:      49152 bytes
  Total number of registers available per block: 65536
  Warp size:                                    32
  Maximum threads per multiprocessor (SM):      1024
  Maximum threads per block:                    1024
  Max dimension size of a thread block (x,y,z): 1024, 1024, 64
  Max dimension size of a grid size    (x,y,z): 2147483647, 65535, 65535
==5746== Profiling application: /content/test
==5746== Profiling result:
No kernels were profiled.
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
      API calls:   51.87%  2.3356ms       114  20.488us      89ns  1.1759ms  cuDeviceGetAtt